# S5Mars PTQ: FP32, FP16 and INT8 on NVIDIA T4

Self-contained Kaggle notebook for post-training quantization of a completed S5Mars
training run. It loads best.ckpt, exports ONNX, builds TensorRT FP32/FP16/INT8
engines, evaluates the existing segmentation metrics, benchmarks batch-size-one
latency, and optionally uploads every artifact to Google Drive.

The notebook intentionally does not train or perform QAT. Selection uses train and
validation data; enable the final test evaluation only after the PTQ configuration is
frozen.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "datasets",
    "huggingface_hub",
    "transformers",
    "segmentation-models-pytorch",
    "onnx",
    "onnxruntime-gpu",
    "onnxconverter-common",
    "nvidia-modelopt[onnx]",
    "google-api-python-client",
    "google-auth",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])

In [ ]:
# Source run. For Drive, provide the exact training run folder ID.
SOURCE_MODE = "drive"  # drive or local
SOURCE_RUN_DIR = None
SOURCE_DRIVE_FOLDER_ID = None

REPO_ID = "Mirali33/mb-s5mars"
TRAIN_SPLIT = "train"
VAL_SPLIT = "val"
TEST_SPLIT = "test"
HF_TOKEN_SECRET = "HF_TOKEN"

OUTPUT_ROOT = "/kaggle/working"
RUN_FP32_TRT = True
RUN_FP16_TRT = True
RUN_INT8_PTQ = True
RUN_FINAL_TEST = False

CALIBRATION_SIZE = 128
CALIBRATION_SEED = 42
CALIBRATION_STRATEGY = "random"  # random or class_coverage
CALIBRATION_METHOD = "entropy"  # entropy or max
CALIBRATION_BATCH_SIZE = 1

BENCHMARK_BATCH_SIZE = 1
BENCHMARK_WARMUP_ITERATIONS = 100
BENCHMARK_MEASUREMENT_ITERATIONS = 1000
BENCHMARK_TRIALS = 3

LIMIT_VAL_BATCHES = None
LIMIT_TEST_BATCHES = None
NUM_WORKERS = 2
ONNX_OPSET = 18
TRTEXEC_PATH = None

DRIVE_UPLOAD_ENABLED = False
DRIVE_PARENT_FOLDER_ID = None
DRIVE_UPLOAD_CHECKPOINTS = True

assert SOURCE_MODE in {"drive", "local"}
assert CALIBRATION_STRATEGY in {"random", "class_coverage"}
assert CALIBRATION_METHOD in {"entropy", "max"}
assert CALIBRATION_BATCH_SIZE == 1, "Static ONNX input fixes calibration batch size to 1"
assert BENCHMARK_BATCH_SIZE == 1

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from onnxconverter_common import float16
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("s5mars_ptq")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("This notebook requires an NVIDIA GPU")
torch.cuda.set_device(0)

def get_kaggle_secret(name, required=False):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle secret: {name}")
    return value

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True, default=str) + "\n")
    os.replace(temporary, path)
    return path

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_run_id(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", value).strip("-")

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def command_output(command):
    try:
        result = subprocess.run(command, check=False, capture_output=True, text=True)
        return {"returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except FileNotFoundError:
        return {"returncode": None, "stdout": "", "stderr": "not found"}

def find_trtexec():
    candidates = [
        TRTEXEC_PATH,
        shutil.which("trtexec"),
        "/usr/src/tensorrt/bin/trtexec",
        "/opt/tensorrt/bin/trtexec",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return str(candidate)
    return None

TRTEXEC = find_trtexec()
ENVIRONMENT = {
    "captured_at_utc": utc_now(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_count": torch.cuda.device_count(),
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "packages": {
        name: package_version(name)
        for name in [
            "onnx", "onnxruntime-gpu", "tensorrt", "nvidia-modelopt",
            "transformers", "segmentation-models-pytorch"
        ]
    },
    "nvidia_smi": command_output(["nvidia-smi", "-q"]),
    "trtexec_version": (
        command_output([TRTEXEC, "--version"])
        if TRTEXEC else {"returncode": None, "stdout": "", "stderr": "not found"}
    ),
}
print(json.dumps({k: v for k, v in ENVIRONMENT.items() if k != "nvidia_smi"}, indent=2))

In [ ]:
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
FOLDER_MIME = "application/vnd.google-apps.folder"

def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)

def drive_children(service, parent_id, name=None):
    escaped_parent = parent_id.replace("'", "\\'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace("\\", "\\\\").replace("'", "\\'")
        query.append(f"name = '{escaped_name}'")
    response = service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size)",
        pageSize=1000,
    ).execute(num_retries=3)
    return response.get("files", [])

def ensure_drive_folder(service, parent_id, name, fail_if_exists=False):
    matches = [item for item in drive_children(service, parent_id, name)
               if item["mimeType"] == FOLDER_MIME]
    if matches:
        if fail_if_exists:
            raise FileExistsError(f"Drive folder already exists: {name}")
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {name}")
        return matches[0]["id"]
    created = service.files().create(
        body={"name": name, "mimeType": FOLDER_MIME, "parents": [parent_id]},
        fields="id",
    ).execute(num_retries=3)
    return created["id"]

def drive_download_file(service, file_id, destination):
    from googleapiclient.http import MediaIoBaseDownload
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with destination.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=5)
    return destination

def drive_download_named(service, folder_id, names, destination):
    by_name = {item["name"]: item for item in drive_children(service, folder_id)}
    missing = sorted(set(names) - by_name.keys())
    if missing:
        raise FileNotFoundError(f"Drive source run is missing: {missing}")
    return {
        name: drive_download_file(service, by_name[name]["id"], Path(destination) / name)
        for name in names
    }

def drive_upload_file(service, path, parent_id):
    from googleapiclient.http import MediaFileUpload
    path = Path(path)
    media = MediaFileUpload(
        str(path), mimetype="application/octet-stream",
        chunksize=8 * 1024 * 1024, resumable=True,
    )
    request = service.files().create(
        body={"name": path.name, "parents": [parent_id]},
        media_body=media,
        fields="id,name,size",
    )
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=5)
    return response

def upload_run_tree(service, root, parent_folder_id, hierarchy):
    parent = parent_folder_id
    for name in hierarchy[:-1]:
        parent = ensure_drive_folder(service, parent, name)
    run_folder = ensure_drive_folder(service, parent, hierarchy[-1], fail_if_exists=True)
    folder_ids = {Path("."): run_folder}
    uploaded, failures = [], []
    root = Path(root)
    for path in sorted(root.rglob("*")):
        relative = path.relative_to(root)
        if path.is_dir():
            folder_ids[relative] = ensure_drive_folder(
                service, folder_ids[relative.parent], path.name
            )
            continue
        if path.name == "upload_status.json":
            continue
        try:
            response = drive_upload_file(service, path, folder_ids[relative.parent])
            uploaded.append({
                "path": relative.as_posix(),
                "drive_file_id": response["id"],
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
        except Exception as error:
            logger.exception("Upload failed for %s", relative)
            failures.append({"path": relative.as_posix(), "error": type(error).__name__})
    return {
        "status": "complete" if not failures else "partial",
        "drive_folder_id": run_folder,
        "uploaded": uploaded,
        "failures": failures,
    }

In [ ]:
HF_TOKEN = get_kaggle_secret(HF_TOKEN_SECRET, required=False)

class SegmentationTransform:
    def __init__(self, size, mean, std):
        self.size = tuple(size)
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1)

    def __call__(self, image, mask):
        image = image.convert("RGB").resize(self.size[::-1], Image.Resampling.BILINEAR)
        mask = mask.convert("L").resize(self.size[::-1], Image.Resampling.NEAREST)
        array = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
        image_tensor = (image_tensor - self.mean) / self.std
        mask_tensor = torch.from_numpy(np.asarray(mask, dtype=np.int64)).long()
        return image_tensor, mask_tensor

class PTQS5MarsDataset(Dataset):
    def __init__(self, repo_id, split, token, transform):
        self.dataset = load_dataset(repo_id, split=split, token=token)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image, mask = self.transform(sample["image"], sample["mask"])
        return {"image": image, "mask": mask, "index": int(index)}

    def image_level_labels(self):
        return self.dataset["class_labels"]

def seeded_indices(dataset_size, sample_size, seed):
    if sample_size <= 0 or sample_size > dataset_size:
        raise ValueError("Invalid calibration size")
    values = list(range(dataset_size))
    random.Random(seed).shuffle(values)
    return values[:sample_size]

def class_coverage_indices(labels_by_index, sample_size, seed):
    label_sets = [set(labels) for labels in labels_by_index]
    permutation = seeded_indices(len(label_sets), len(label_sets), seed)
    rank = {index: position for position, index in enumerate(permutation)}
    uncovered = set().union(*label_sets) if label_sets else set()
    remaining = set(range(len(label_sets)))
    selected = []
    while uncovered and remaining and len(selected) < sample_size:
        best = max(
            remaining,
            key=lambda index: (len(label_sets[index] & uncovered), -rank[index]),
        )
        if not (label_sets[best] & uncovered):
            break
        selected.append(best)
        remaining.remove(best)
        uncovered.difference_update(label_sets[best])
    for index in permutation:
        if len(selected) == sample_size:
            break
        if index in remaining:
            selected.append(index)
            remaining.remove(index)
    return selected

In [ ]:
from transformers import SegformerForSemanticSegmentation
import segmentation_models_pytorch as smp

class SegFormerForMars(nn.Module):
    def __init__(self, pretrained_name, num_classes, ignore_index):
        super().__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            pretrained_name,
            num_labels=num_classes,
            semantic_loss_ignore_index=ignore_index,
            ignore_mismatched_sizes=True,
        )

    def forward(self, images):
        logits = self.model(pixel_values=images).logits
        return F.interpolate(
            logits, size=images.shape[-2:], mode="bilinear", align_corners=False
        )

class SMPModelForMars(nn.Module):
    def __init__(self, architecture, encoder_name, encoder_weights, in_channels, num_classes):
        super().__init__()
        constructors = {
            "unet": smp.Unet,
            "deeplabv3": smp.DeepLabV3,
            "deeplabv3plus": smp.DeepLabV3Plus,
        }
        self.model = constructors[architecture.lower()](
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=num_classes,
            activation=None,
        )

    def forward(self, images):
        return self.model(images)

class ConvBNSelu(nn.Module):
    def __init__(
        self, in_channels, out_channels, kernel_size, stride=1, padding=0,
        dilation=1, groups=1, norm_activation=True,
    ):
        super().__init__()
        layers = [
            nn.Conv2d(
                in_channels, out_channels, kernel_size, stride=stride,
                padding=padding, dilation=dilation, groups=groups, bias=not norm_activation,
            )
        ]
        if norm_activation:
            layers += [nn.BatchNorm2d(out_channels, eps=1e-3), nn.SELU(inplace=True)]
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

class DownsamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        if out_channels <= in_channels:
            raise ValueError("out_channels must exceed in_channels")
        self.conv = ConvBNSelu(
            in_channels, out_channels - in_channels, 3, stride=2,
            padding=1, norm_activation=False,
        )
        self.pool = nn.MaxPool2d(2, stride=2)
        self.norm_activation = nn.Sequential(
            nn.BatchNorm2d(out_channels, eps=1e-3), nn.SELU(inplace=True)
        )

    def forward(self, x):
        return self.norm_activation(torch.cat((self.conv(x), self.pool(x)), dim=1))

class ThreeBranchContextAggregation(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.base = ConvBNSelu(channels, channels, 3, padding=1)
        self.local_depthwise = ConvBNSelu(
            channels, channels, 3, padding=1, groups=channels
        )
        self.dilated_depthwise = ConvBNSelu(
            channels, channels, 3, padding=dilation,
            dilation=dilation, groups=channels,
        )
        self.norm_activation = nn.Sequential(
            nn.BatchNorm2d(channels, eps=1e-3), nn.SELU(inplace=True)
        )

    def forward(self, x):
        base = self.base(x)
        return self.norm_activation(
            base + self.local_depthwise(base) + self.dilated_depthwise(base)
        )

class PartialChannelTransformation(nn.Module):
    def __init__(self, channels, dilation, partial_rate):
        super().__init__()
        self.identity_channels = round(channels * (1.0 - partial_rate))
        transformed_channels = channels - self.identity_channels
        self.tca = ThreeBranchContextAggregation(transformed_channels, dilation)
        self.pointwise = ConvBNSelu(channels, channels, 1)

    def forward(self, x):
        identity, transformed = torch.split(
            x, (self.identity_channels, x.shape[1] - self.identity_channels), dim=1
        )
        return self.pointwise(torch.cat((identity, self.tca(transformed)), dim=1))

class DualAttentionGuidedDecoder(nn.Module):
    def __init__(self, deep_channels, shallow_channels, num_classes):
        super().__init__()
        self.deep_projection = ConvBNSelu(deep_channels, shallow_channels, 1)
        self.negative_projection = ConvBNSelu(shallow_channels, shallow_channels, 1)
        self.depthwise = ConvBNSelu(
            shallow_channels, shallow_channels, 3, padding=1, groups=shallow_channels
        )
        self.classifier = ConvBNSelu(shallow_channels, num_classes, 1)

    def forward(self, shallow, deep):
        spatial_map = torch.sigmoid(shallow)
        projected = self.deep_projection(deep)
        projected_map = torch.sigmoid(projected)
        reverse_guided = self.negative_projection(-projected_map) * projected_map + projected
        reverse_guided = F.interpolate(
            reverse_guided, size=spatial_map.shape[-2:],
            mode="bilinear", align_corners=False,
        )
        return self.classifier(self.depthwise(spatial_map * reverse_guided))

LCNET_DILATIONS = {
    "lcnet3_7": ((1, 2, 5), (1, 2, 5, 9, 2, 5, 9)),
    "lcnet3_11": ((1, 2, 5), (1, 2, 5, 9, 2, 5, 9, 2, 5, 9, 17)),
}

class LCNet(nn.Module):
    def __init__(
        self, variant, in_channels, base_channels, partial_rate,
        stage1_blocks, stage2_blocks, num_classes,
    ):
        super().__init__()
        stage1_dilations, stage2_dilations = LCNET_DILATIONS[variant]
        if (stage1_blocks, stage2_blocks) != (
            len(stage1_dilations), len(stage2_dilations)
        ):
            raise ValueError("LCNet stage configuration does not match its variant")
        self.initialization = nn.Sequential(
            ConvBNSelu(in_channels, base_channels, 3, stride=2, padding=1),
            ConvBNSelu(base_channels, base_channels, 3, padding=1),
            ConvBNSelu(base_channels, base_channels, 3, padding=1),
        )
        self.stage1 = self._stage(
            base_channels, base_channels * 2, stage1_dilations, partial_rate
        )
        self.stage2 = self._stage(
            base_channels * 2, base_channels * 4, stage2_dilations, partial_rate
        )
        self.decoder = DualAttentionGuidedDecoder(
            base_channels * 4, base_channels * 2, num_classes
        )

    @staticmethod
    def _stage(in_channels, out_channels, dilations, partial_rate):
        return nn.Sequential(
            DownsamplingBlock(in_channels, out_channels),
            *(PartialChannelTransformation(out_channels, value, partial_rate)
              for value in dilations),
        )

    def forward(self, images):
        input_size = images.shape[-2:]
        shallow = self.stage1(self.initialization(images))
        deep = self.stage2(shallow)
        return F.interpolate(
            self.decoder(shallow, deep), size=input_size,
            mode="bilinear", align_corners=False,
        )

def build_model(model_cfg):
    name = model_cfg["name"]
    if name == "segformer_b0":
        return SegFormerForMars(
            model_cfg["pretrained_name"],
            model_cfg["num_classes"],
            model_cfg["ignore_index"],
        )
    if name == "smp":
        return SMPModelForMars(
            model_cfg["architecture"],
            model_cfg["encoder_name"],
            model_cfg.get("encoder_weights"),
            model_cfg.get("in_channels", 3),
            model_cfg["num_classes"],
        )
    if name == "lcnet":
        return LCNet(
            model_cfg["variant"],
            model_cfg.get("in_channels", 3),
            model_cfg.get("base_channels", 32),
            model_cfg.get("partial_rate", 0.5),
            model_cfg["stage1_blocks"],
            model_cfg["stage2_blocks"],
            model_cfg["num_classes"],
        )
    raise ValueError(f"Unknown model name: {name}")

In [ ]:
def create_confusion_matrix(num_classes, device):
    return torch.zeros((num_classes, num_classes), dtype=torch.float64, device=device)

@torch.no_grad()
def update_confusion_matrix(confmat, preds, targets, num_classes, ignore_index):
    preds = preds.reshape(-1)
    targets = targets.reshape(-1)
    valid = targets != ignore_index
    preds = preds[valid]
    targets = targets[valid]
    valid_range = (targets >= 0) & (targets < num_classes)
    preds = preds[valid_range]
    targets = targets[valid_range]
    indices = targets * num_classes + preds
    confmat += torch.bincount(
        indices, minlength=num_classes * num_classes
    ).reshape(num_classes, num_classes).to(confmat.dtype)
    return confmat

def compute_segmentation_metrics(confmat, ignore_index=0):
    tp = torch.diag(confmat)
    support = confmat.sum(dim=1)
    predicted = confmat.sum(dim=0)
    union = support + predicted - tp
    iou = tp / torch.clamp(union, min=1.0)
    valid_classes = torch.ones_like(iou, dtype=torch.bool)
    if 0 <= ignore_index < len(iou):
        valid_classes[ignore_index] = False
    valid_classes = valid_classes & (support > 0)
    miou = (
        iou[valid_classes].mean()
        if valid_classes.any()
        else torch.tensor(0.0, device=confmat.device)
    )
    pixel_accuracy = tp.sum() / torch.clamp(confmat.sum(), min=1.0)
    return {
        "pixel_accuracy": pixel_accuracy.item(),
        "miou": miou.item(),
        "per_class_iou": iou.detach().cpu().tolist(),
    }

@torch.no_grad()
def evaluate_callable(runtime, loader, criterion, num_classes, ignore_index, desc, limit=None):
    total_loss = 0.0
    batches = 0
    confmat = create_confusion_matrix(num_classes, DEVICE)
    for batch_index, batch in enumerate(tqdm(loader, desc=desc)):
        if limit is not None and batch_index >= limit:
            break
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        logits = runtime(images)
        logits_fp32 = logits.float()
        total_loss += criterion(logits_fp32, masks).item()
        preds = torch.argmax(logits_fp32, dim=1)
        confmat = update_confusion_matrix(
            confmat, preds, masks, num_classes, ignore_index
        )
        batches += 1
    result = compute_segmentation_metrics(confmat, ignore_index)
    result["loss"] = total_loss / max(batches, 1)
    return result

In [ ]:
SOURCE_CACHE = Path(OUTPUT_ROOT) / "_source_run"
SOURCE_CACHE.mkdir(parents=True, exist_ok=True)

if SOURCE_MODE == "drive":
    if not SOURCE_DRIVE_FOLDER_ID:
        raise ValueError("Set SOURCE_DRIVE_FOLDER_ID")
    DRIVE_SERVICE = build_drive_service()
    source_files = drive_download_named(
        DRIVE_SERVICE,
        SOURCE_DRIVE_FOLDER_ID,
        {"run_manifest.json", "best.ckpt"},
        SOURCE_CACHE,
    )
    source_dir = SOURCE_CACHE
else:
    source_dir = Path(SOURCE_RUN_DIR)
    if not source_dir.is_dir():
        raise FileNotFoundError(source_dir)

source_manifest = json.loads((source_dir / "run_manifest.json").read_text())
if source_manifest.get("schema_version") != 1:
    raise ValueError("Unsupported source run manifest schema")
source_run_id = source_manifest["run_id"]
model_run_name = source_manifest["model_run_name"]
model_cfg = source_manifest["config"]["model"]
preprocessing_cfg = source_manifest["config"]["preprocessing"]
expected_sha = source_manifest.get("best_checkpoint", {}).get("sha256")
checkpoint_path = source_dir / "best.ckpt"
if expected_sha and sha256_file(checkpoint_path) != expected_sha:
    raise ValueError("best.ckpt SHA-256 does not match run_manifest.json")

RUN_ID = safe_run_id(
    f"{model_run_name}__ptq__calib{CALIBRATION_SIZE}_{CALIBRATION_STRATEGY}"
    f"__seed{CALIBRATION_SEED}__{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)
OUTPUT_DIR = Path(OUTPUT_ROOT) / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
save_json(ENVIRONMENT, OUTPUT_DIR / "environment.json")

model = build_model(model_cfg).to(DEVICE).eval()
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
sample = torch.randn(
    (1, 3, *preprocessing_cfg["image_size"]), dtype=torch.float32, device=DEVICE
)
with torch.inference_mode():
    smoke_logits = model(sample)
assert tuple(smoke_logits.shape) == (
    1, model_cfg["num_classes"], *preprocessing_cfg["image_size"]
)
assert torch.isfinite(smoke_logits).all()

transform = SegmentationTransform(
    preprocessing_cfg["image_size"],
    preprocessing_cfg["mean"],
    preprocessing_cfg["std"],
)
train_dataset = PTQS5MarsDataset(REPO_ID, TRAIN_SPLIT, HF_TOKEN, transform)
val_dataset = PTQS5MarsDataset(REPO_ID, VAL_SPLIT, HF_TOKEN, transform)
test_dataset = PTQS5MarsDataset(REPO_ID, TEST_SPLIT, HF_TOKEN, transform)

if CALIBRATION_STRATEGY == "random":
    calibration_indices = seeded_indices(
        len(train_dataset), CALIBRATION_SIZE, CALIBRATION_SEED
    )
else:
    calibration_indices = class_coverage_indices(
        train_dataset.image_level_labels(), CALIBRATION_SIZE, CALIBRATION_SEED
    )

calibration_loader = DataLoader(
    Subset(train_dataset, calibration_indices),
    batch_size=CALIBRATION_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
calibration_manifest = {
    "repo_id": REPO_ID,
    "split": TRAIN_SPLIT,
    "strategy": CALIBRATION_STRATEGY,
    "seed": CALIBRATION_SEED,
    "size": len(calibration_indices),
    "batch_size": CALIBRATION_BATCH_SIZE,
    "indices": calibration_indices,
    "indices_sha256": hashlib.sha256(
        json.dumps(calibration_indices).encode("utf-8")
    ).hexdigest(),
}
save_json(calibration_manifest, OUTPUT_DIR / "calibration_manifest.json")
criterion = nn.CrossEntropyLoss(ignore_index=model_cfg["ignore_index"])

In [ ]:
FP32_ONNX = OUTPUT_DIR / "model_fp32.onnx"
FP16_ONNX = OUTPUT_DIR / "model_fp16.onnx"
INT8_ONNX = OUTPUT_DIR / "model_int8_qdq.onnx"
INPUT_NAME = "input"
OUTPUT_NAME = "logits"

with torch.inference_mode():
    torch.onnx.export(
        model,
        sample,
        FP32_ONNX,
        input_names=[INPUT_NAME],
        output_names=[OUTPUT_NAME],
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        dynamo=False,
    )
onnx_model = onnx.load(FP32_ONNX)
onnx.checker.check_model(onnx_model)
onnx.save(onnx.shape_inference.infer_shapes(onnx_model), FP32_ONNX)

import onnxruntime as ort
providers = [
    provider for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if provider in ort.get_available_providers()
]
ort_session = ort.InferenceSession(str(FP32_ONNX), providers=providers)
real_sample = next(iter(val_loader))["image"].to(DEVICE)
with torch.inference_mode():
    torch_output = model(real_sample).detach().cpu().numpy()
ort_output = ort_session.run(
    [OUTPUT_NAME], {INPUT_NAME: real_sample.cpu().numpy()}
)[0]
difference = np.abs(torch_output - ort_output)
export_validation = {
    "providers": ort_session.get_providers(),
    "mean_absolute_error": float(difference.mean()),
    "max_absolute_error": float(difference.max()),
    "pixel_agreement": float(
        (torch_output.argmax(1) == ort_output.argmax(1)).mean()
    ),
    "output_shape": list(ort_output.shape),
}
save_json(export_validation, OUTPUT_DIR / "export_validation.json")
if export_validation["pixel_agreement"] < 0.99:
    raise RuntimeError("FP32 ONNX export failed the pixel-agreement gate")

pytorch_validation_metrics = evaluate_callable(
    model, val_loader, criterion, model_cfg["num_classes"],
    model_cfg["ignore_index"], "Validation PyTorch FP32", LIMIT_VAL_BATCHES,
)
save_json(
    pytorch_validation_metrics,
    OUTPUT_DIR / "val_metrics_pytorch_fp32.json",
)
if RUN_FINAL_TEST:
    pytorch_test_metrics = evaluate_callable(
        model, test_loader, criterion, model_cfg["num_classes"],
        model_cfg["ignore_index"], "Test PyTorch FP32", LIMIT_TEST_BATCHES,
    )
    save_json(
        pytorch_test_metrics,
        OUTPUT_DIR / "test_metrics_pytorch_fp32.json",
    )

if RUN_FP16_TRT:
    fp16_model = float16.convert_float_to_float16(
        onnx.load(FP32_ONNX), keep_io_types=True
    )
    onnx.checker.check_model(fp16_model)
    onnx.save(fp16_model, FP16_ONNX)

In [ ]:
if RUN_INT8_PTQ:
    from onnxruntime.quantization import CalibrationDataReader
    from modelopt.onnx.quantization import quantize as modelopt_quantize

    class LoaderCalibrationReader(CalibrationDataReader):
        def __init__(self, loader):
            self.loader = loader
            self.rewind()

        def get_next(self):
            try:
                batch = next(self.iterator)
            except StopIteration:
                return None
            return {INPUT_NAME: batch["image"].numpy().astype(np.float32)}

        def rewind(self):
            self.iterator = iter(self.loader)

    calibration_reader = LoaderCalibrationReader(calibration_loader)
    modelopt_quantize(
        str(FP32_ONNX),
        quantize_mode="int8",
        calibration_data_reader=calibration_reader,
        calibration_method=CALIBRATION_METHOD,
        output_path=str(INT8_ONNX),
        high_precision_dtype="fp16",
        override_shapes={INPUT_NAME: [1, 3, *preprocessing_cfg["image_size"]]},
        calibration_eps=["cuda:0", "cpu"],
        use_external_data_format=False,
    )
    int8_graph = onnx.load(INT8_ONNX)
    onnx.checker.check_model(int8_graph)
    op_counts = {}
    for node in int8_graph.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
    quantization_summary = {
        "method": "ptq",
        "format": "int8_w8a8_qdq",
        "calibration_method": CALIBRATION_METHOD,
        "quantize_linear_nodes": op_counts.get("QuantizeLinear", 0),
        "dequantize_linear_nodes": op_counts.get("DequantizeLinear", 0),
        "operator_counts": op_counts,
    }
    save_json(quantization_summary, OUTPUT_DIR / "quantization_summary.json")
    if not quantization_summary["quantize_linear_nodes"]:
        raise RuntimeError("INT8 ONNX graph contains no QuantizeLinear nodes")

model.to("cpu")
del smoke_logits, sample, real_sample
torch.cuda.empty_cache()

In [ ]:
import tensorrt as trt

def trtexec_supports(flag):
    if not TRTEXEC:
        return False
    help_result = subprocess.run(
        [TRTEXEC, "--help"], capture_output=True, text=True, check=False
    )
    return flag in (help_result.stdout + help_result.stderr)

def build_engine_python(onnx_path, engine_path, log_path):
    trt_logger = trt.Logger(trt.Logger.INFO)
    builder = trt.Builder(trt_logger)
    flags = 0
    if hasattr(trt.NetworkDefinitionCreationFlag, "STRONGLY_TYPED"):
        flags |= 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)
    elif hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH"):
        flags |= 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, trt_logger)
    parsed = parser.parse(Path(onnx_path).read_bytes())
    if not parsed:
        errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
        Path(log_path).write_text("\n".join(errors), encoding="utf-8")
        raise RuntimeError(f"TensorRT ONNX parse failed; inspect {log_path}")
    config = builder.create_builder_config()
    if hasattr(config, "builder_optimization_level"):
        config.builder_optimization_level = 5
    if hasattr(config, "set_memory_pool_limit"):
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    if hasattr(builder, "build_serialized_network"):
        serialized = builder.build_serialized_network(network, config)
    else:
        engine = builder.build_engine(network, config)
        serialized = engine.serialize() if engine is not None else None
    if serialized is None:
        raise RuntimeError("TensorRT Python builder returned no engine")
    Path(engine_path).write_bytes(bytes(serialized))
    Path(log_path).write_text(
        "Engine built with TensorRT Python API\n", encoding="utf-8"
    )
    return {"backend": "python", "returncode": 0}

def build_engine(onnx_path, engine_path, log_path):
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    if not TRTEXEC:
        return build_engine_python(onnx_path, engine_path, log_path)
    command = [
        TRTEXEC,
        f"--onnx={onnx_path}",
        f"--saveEngine={engine_path}",
        f"--shapes={INPUT_NAME}:1x3x{preprocessing_cfg['image_size'][0]}x"
        f"{preprocessing_cfg['image_size'][1]}",
        "--builderOptimizationLevel=5",
    ]
    if trtexec_supports("--skipInference"):
        command.append("--skipInference")
    if trtexec_supports("--stronglyTyped"):
        command.append("--stronglyTyped")
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    Path(log_path).write_text(result.stdout + "\n" + result.stderr)
    if result.returncode != 0 or not Path(engine_path).is_file():
        raise RuntimeError(f"TensorRT build failed; inspect {log_path}")
    return {"backend": "trtexec", "command": command, "returncode": result.returncode}

ENGINE_PATHS = {}
BUILD_RESULTS = {}
if RUN_FP32_TRT:
    ENGINE_PATHS["fp32"] = OUTPUT_DIR / "engine_fp32.plan"
    BUILD_RESULTS["fp32"] = build_engine(
        FP32_ONNX, ENGINE_PATHS["fp32"], OUTPUT_DIR / "logs/build_fp32.log"
    )
if RUN_FP16_TRT:
    ENGINE_PATHS["fp16"] = OUTPUT_DIR / "engine_fp16.plan"
    BUILD_RESULTS["fp16"] = build_engine(
        FP16_ONNX, ENGINE_PATHS["fp16"], OUTPUT_DIR / "logs/build_fp16.log"
    )
if RUN_INT8_PTQ:
    ENGINE_PATHS["int8"] = OUTPUT_DIR / "engine_int8.plan"
    BUILD_RESULTS["int8"] = build_engine(
        INT8_ONNX, ENGINE_PATHS["int8"], OUTPUT_DIR / "logs/build_int8.log"
    )
save_json(BUILD_RESULTS, OUTPUT_DIR / "engine_build_results.json")

TRT_TO_TORCH = {
    trt.float32: torch.float32,
    trt.float16: torch.float16,
    trt.int8: torch.int8,
    trt.int32: torch.int32,
    trt.bool: torch.bool,
}

class TensorRTRunner:
    def __init__(self, engine_path):
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        self.engine = self.runtime.deserialize_cuda_engine(
            Path(engine_path).read_bytes()
        )
        if self.engine is None:
            raise RuntimeError(f"Could not deserialize {engine_path}")
        self.context = self.engine.create_execution_context()
        self.stream = torch.cuda.Stream(device=DEVICE)
        self.v3 = hasattr(self.engine, "num_io_tensors")
        self.buffers = {}
        self.input_name = None
        self.output_names = []
        if self.v3:
            for index in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(index)
                mode = self.engine.get_tensor_mode(name)
                if mode == trt.TensorIOMode.INPUT:
                    self.input_name = name
                    self.context.set_input_shape(
                        name, (1, 3, *preprocessing_cfg["image_size"])
                    )
            for index in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(index)
                shape = tuple(self.context.get_tensor_shape(name))
                dtype = TRT_TO_TORCH[self.engine.get_tensor_dtype(name)]
                self.buffers[name] = torch.empty(shape, dtype=dtype, device=DEVICE)
                self.context.set_tensor_address(name, self.buffers[name].data_ptr())
                if self.engine.get_tensor_mode(name) == trt.TensorIOMode.OUTPUT:
                    self.output_names.append(name)
        else:
            self.bindings = [None] * self.engine.num_bindings
            for index in range(self.engine.num_bindings):
                if self.engine.binding_is_input(index):
                    self.input_name = self.engine.get_binding_name(index)
                    self.context.set_binding_shape(
                        index, (1, 3, *preprocessing_cfg["image_size"])
                    )
            for index in range(self.engine.num_bindings):
                name = self.engine.get_binding_name(index)
                shape = tuple(self.context.get_binding_shape(index))
                dtype = TRT_TO_TORCH[self.engine.get_binding_dtype(index)]
                self.buffers[name] = torch.empty(shape, dtype=dtype, device=DEVICE)
                self.bindings[index] = self.buffers[name].data_ptr()
                if not self.engine.binding_is_input(index):
                    self.output_names.append(name)

    def infer(self, images, synchronize=True, clone_output=True):
        input_buffer = self.buffers[self.input_name]
        with torch.cuda.stream(self.stream):
            input_buffer.copy_(images.to(dtype=input_buffer.dtype), non_blocking=True)
            if self.v3:
                ok = self.context.execute_async_v3(self.stream.cuda_stream)
            else:
                ok = self.context.execute_async_v2(
                    self.bindings, self.stream.cuda_stream
                )
        if not ok:
            raise RuntimeError("TensorRT execution failed")
        if synchronize:
            self.stream.synchronize()
        output = self.buffers[self.output_names[0]]
        return output.clone() if clone_output else output

In [ ]:
def percentile(sorted_values, q):
    position = (len(sorted_values) - 1) * q
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return sorted_values[lower]
    weight = position - lower
    return sorted_values[lower] * (1 - weight) + sorted_values[upper] * weight

def summarize_timings(values):
    values = [float(value) for value in values]
    ordered = sorted(values)
    mean = sum(values) / len(values)
    variance = sum((value - mean) ** 2 for value in values) / len(values)
    median = percentile(ordered, 0.5)
    return {
        "num_measurements": len(values),
        "min_latency_ms": ordered[0],
        "max_latency_ms": ordered[-1],
        "mean_latency_ms": mean,
        "latency_std_ms": math.sqrt(variance),
        "median_latency_ms": median,
        "p90_latency_ms": percentile(ordered, 0.90),
        "p95_latency_ms": percentile(ordered, 0.95),
        "p99_latency_ms": percentile(ordered, 0.99),
        "fps_from_mean": 1000.0 / mean,
        "fps_from_median": 1000.0 / median,
    }

def benchmark_runner(runner, images, warmup, measurements):
    for _ in range(warmup):
        runner.infer(images, synchronize=False, clone_output=False)
    runner.stream.synchronize()
    events = []
    for _ in range(measurements):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record(runner.stream)
        runner.infer(images, synchronize=False, clone_output=False)
        end.record(runner.stream)
        events.append((start, end))
    events[-1][1].synchronize()
    return summarize_timings([start.elapsed_time(end) for start, end in events])

runners = {name: TensorRTRunner(path) for name, path in ENGINE_PATHS.items()}
validation_metrics = {}
test_metrics = {}
benchmark_results = {"batch_size": 1, "trials": {}}
benchmark_images = next(iter(test_loader))["image"].to(DEVICE)

for precision, runner in runners.items():
    validation_metrics[precision] = evaluate_callable(
        runner.infer,
        val_loader,
        criterion,
        model_cfg["num_classes"],
        model_cfg["ignore_index"],
        f"Validation {precision}",
        LIMIT_VAL_BATCHES,
    )
    save_json(
        validation_metrics[precision],
        OUTPUT_DIR / f"val_metrics_{precision}.json",
    )
    if RUN_FINAL_TEST:
        test_metrics[precision] = evaluate_callable(
            runner.infer,
            test_loader,
            criterion,
            model_cfg["num_classes"],
            model_cfg["ignore_index"],
            f"Test {precision}",
            LIMIT_TEST_BATCHES,
        )
        save_json(
            test_metrics[precision],
            OUTPUT_DIR / f"test_metrics_{precision}.json",
        )
    benchmark_results["trials"][precision] = [
        benchmark_runner(
            runner,
            benchmark_images,
            BENCHMARK_WARMUP_ITERATIONS,
            BENCHMARK_MEASUREMENT_ITERATIONS,
        )
        for _ in range(BENCHMARK_TRIALS)
    ]

baseline_trials = benchmark_results["trials"].get("fp32")
if baseline_trials:
    baseline_median = sum(
        trial["median_latency_ms"] for trial in baseline_trials
    ) / len(baseline_trials)
    for precision, trials in benchmark_results["trials"].items():
        candidate = sum(trial["median_latency_ms"] for trial in trials) / len(trials)
        for trial in trials:
            trial["speedup_vs_trt_fp32"] = baseline_median / candidate
save_json(benchmark_results, OUTPUT_DIR / "benchmark_results.json")

comparison = {}
if "fp32" in test_metrics:
    baseline = test_metrics["fp32"]
    for precision, metrics_value in test_metrics.items():
        comparison[precision] = {
            "delta_miou_vs_trt_fp32": metrics_value["miou"] - baseline["miou"],
            "delta_pixel_accuracy_vs_trt_fp32": (
                metrics_value["pixel_accuracy"] - baseline["pixel_accuracy"]
            ),
            "delta_per_class_iou_vs_trt_fp32": [
                current - reference
                for current, reference in zip(
                    metrics_value["per_class_iou"], baseline["per_class_iou"]
                )
            ],
        }
save_json(comparison, OUTPUT_DIR / "comparison_metrics.json")

In [ ]:
run_manifest = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "run_type": "ptq",
    "created_at_utc": utc_now(),
    "model_run_name": model_run_name,
    "seed": CALIBRATION_SEED,
    "source_run_id": source_run_id,
    "source_checkpoint_sha256": sha256_file(checkpoint_path),
    "config": {
        "model": model_cfg,
        "preprocessing": preprocessing_cfg,
        "calibration": calibration_manifest,
        "calibration_method": CALIBRATION_METHOD,
        "benchmark": {
            "batch_size": BENCHMARK_BATCH_SIZE,
            "warmup_iterations": BENCHMARK_WARMUP_ITERATIONS,
            "measurement_iterations": BENCHMARK_MEASUREMENT_ITERATIONS,
            "trials": BENCHMARK_TRIALS,
        },
    },
}
save_json(run_manifest, OUTPUT_DIR / "run_manifest.json")

def build_artifacts_manifest(root):
    root = Path(root)
    artifacts = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.name not in {
            "artifacts_manifest.json", "upload_status.json"
        }:
            artifacts.append({
                "path": path.relative_to(root).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
    return {
        "schema_version": 1,
        "created_at_utc": utc_now(),
        "root_name": root.name,
        "artifacts": artifacts,
    }

save_json(build_artifacts_manifest(OUTPUT_DIR), OUTPUT_DIR / "artifacts_manifest.json")

upload_status = {"status": "disabled"}
if DRIVE_UPLOAD_ENABLED:
    service = globals().get("DRIVE_SERVICE") or build_drive_service()
    parent_id = (
        DRIVE_PARENT_FOLDER_ID
        or get_kaggle_secret("GDRIVE_FOLDER_ID", required=True)
    )
    upload_status = upload_run_tree(
        service,
        OUTPUT_DIR,
        parent_id,
        ["ptq", model_run_name, source_run_id, RUN_ID],
    )
save_json(upload_status, OUTPUT_DIR / "upload_status.json")
if DRIVE_UPLOAD_ENABLED:
    drive_upload_file(
        service,
        OUTPUT_DIR / "upload_status.json",
        upload_status["drive_folder_id"],
    )
print(f"PTQ outputs: {OUTPUT_DIR}")
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))